# Set Transformer: MAB, SAB, ISAB ve PMA Açıklaması

Bu notebook, Set Transformer paper'ındaki özellikle şu kısımları adım adım açıklar:

- **MAB — Multihead Attention Block**
- **SAB — Set Attention Block**
- **ISAB — Induced Set Attention Block**
- **PMA — Pooling by Multihead Attention**
- PMA'nın mean/max pooling yerine neden kullanıldığı
- Seed vector mantığı
- `k` seed ile `k` output üretme
- PMA sonrası SAB ile outputlar arası etkileşim

Amaç: teoriyi hem şematik hem de PyTorch kodu ile somutlaştırmak.


## 1. Set Transformer'da genel akış

Set Transformer, sıralı token sequence yerine **set** işler.

Bir input set şöyle yazılabilir:

$$
X = \{x_1, x_2, \ldots, x_n\}
$$

Burada sıra önemli değildir:

$$
\{x_1, x_2, x_3\} = \{x_3, x_1, x_2\}
$$

Genel mimari:

```text
Input set X
     ↓
Set Encoder: SAB / ISAB
     ↓
Z = {z1, ..., zn}
     ↓
Set Decoder: PMA + optional SAB
     ↓
Output
```

Buradaki `Z`, encoder'dan çıkan contextualized set representation'dır.


## 2. MAB: Multihead Attention Block

Paper'daki temel blok **MAB(X, Y)** şeklinde tanımlanır.

Sezgisel olarak:

> **MAB(X, Y): X tarafındaki elemanlar, Y tarafındaki elemanlardan bilgi çeker.**

Attention açısından:

$$
Q = X,
\quad K = Y,
\quad V = Y
$$

Yani MAB, cross-attention'ın genel formuna benzer. Eğer `X = Y` olursa self-attention olur.

MAB'in iç yapısı:

```text
X ───────────────┐
                 ↓
        Multihead Attention(X, Y, Y)
                 ↓
        Residual + LayerNorm
                 ↓
                 H
                 ↓
        Feed Forward
                 ↓
        Residual + LayerNorm
                 ↓
              Output

Y ───────────────┘
```

Formül olarak:

$$
H = \mathrm{LayerNorm}(X + \mathrm{Multihead}(X, Y, Y))
$$

$$
\mathrm{MAB}(X, Y) = \mathrm{LayerNorm}(H + \mathrm{rFF}(H))
$$

Buradaki `rFF`, row-wise feed-forward network'tür. Yani her set elemanına ayrı ayrı uygulanır.


## 3. Multi-head attention formülü

Paper'daki multi-head attention formu:

$$
\mathrm{Multihead}(Q, K, V) = \mathrm{concat}(O_1, \ldots, O_h) W^O
$$

Her head için:

$$
O_j = \mathrm{Att}(QW_j^Q, KW_j^K, VW_j^V)
$$

Scaled dot-product attention:

$$
\mathrm{Att}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_h}}\right)V
$$

Burada:

- `h`: head sayısı
- `d_h`: head dimension
- `W_j^Q`, `W_j^K`, `W_j^V`: her head'in öğrenilebilir projection matrisleri
- `W^O`: head'leri birleştirdikten sonraki output projection


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Reproducibility
torch.manual_seed(42)

print(torch.__version__)

2.11.0


In [2]:
class MultiHeadAttention(nn.Module):
    """
    Basit Multi-Head Attention implementasyonu.

    Input shapes:
        Q: [batch, n_q, d_model]
        K: [batch, n_k, d_model]
        V: [batch, n_k, d_model]

    Output:
        out:     [batch, n_q, d_model]
        weights: [batch, num_heads, n_q, n_k]
    """
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        # [batch, n, d_model] -> [batch, num_heads, n, d_head]
        batch_size, n, _ = x.shape
        x = x.view(batch_size, n, self.num_heads, self.d_head)
        return x.transpose(1, 2)

    def _combine_heads(self, x: torch.Tensor) -> torch.Tensor:
        # [batch, num_heads, n, d_head] -> [batch, n, d_model]
        batch_size, _, n, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(batch_size, n, self.d_model)

    def forward(self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor):
        q = self._split_heads(self.W_q(Q))
        k = self._split_heads(self.W_k(K))
        v = self._split_heads(self.W_v(V))

        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.d_head)
        weights = F.softmax(scores, dim=-1)
        out = torch.matmul(weights, v)

        out = self._combine_heads(out)
        out = self.W_o(out)
        return out, weights

In [3]:
class RowWiseFeedForward(nn.Module):
    """
    Row-wise FFN.
    Shape'i korur: [batch, n, d_model] -> [batch, n, d_model]
    Her eleman/row için aynı FFN uygulanır.
    """
    def __init__(self, d_model: int, hidden_dim: int = None):
        super().__init__()
        hidden_dim = hidden_dim or 4 * d_model
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class MAB(nn.Module):
    """
    Multihead Attention Block.

    MAB(X, Y):
        Q = X
        K = Y
        V = Y

    H = LayerNorm(X + Multihead(X, Y, Y))
    O = LayerNorm(H + rFF(H))
    """
    def __init__(self, d_model: int, num_heads: int, ff_hidden_dim: int = None):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ff = RowWiseFeedForward(d_model, ff_hidden_dim)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, X: torch.Tensor, Y: torch.Tensor, return_attention: bool = False):
        attn_out, weights = self.attn(X, Y, Y)
        H = self.norm1(X + attn_out)
        O = self.norm2(H + self.ff(H))
        if return_attention:
            return O, weights
        return O

## 4. SAB: Set Attention Block

SAB, MAB'in self-attention halidir:

$$
\mathrm{SAB}(X) = \mathrm{MAB}(X, X)
$$

Yani:

$$
Q = X, \quad K = X, \quad V = X
$$

Her set elemanı diğer set elemanlarına bakar.

```text
Input set X = {x1, x2, ..., xn}
        ↓
SAB(X) = MAB(X, X)
        ↓
Contextualized set representations
```

SAB genellikle **permutation equivariant** davranır. Input sırası değişirse output sırası da aynı şekilde değişir.


In [4]:
class SAB(nn.Module):
    """
    Set Attention Block.
    SAB(X) = MAB(X, X)
    """
    def __init__(self, d_model: int, num_heads: int, ff_hidden_dim: int = None):
        super().__init__()
        self.mab = MAB(d_model, num_heads, ff_hidden_dim)

    def forward(self, X: torch.Tensor, return_attention: bool = False):
        return self.mab(X, X, return_attention=return_attention)

In [5]:
# SAB shape testi
batch_size = 2
n = 5
d_model = 16
num_heads = 4

X = torch.randn(batch_size, n, d_model)
sab = SAB(d_model=d_model, num_heads=num_heads)

out, weights = sab(X, return_attention=True)
print("X shape:      ", X.shape)
print("SAB out shape:", out.shape)
print("Attn weights: ", weights.shape)  # [batch, heads, n_query, n_key]

X shape:       torch.Size([2, 5, 16])
SAB out shape: torch.Size([2, 5, 16])
Attn weights:  torch.Size([2, 4, 5, 5])


## 5. ISAB: Induced Set Attention Block

SAB'in maliyeti yaklaşık:

$$
O(n^2)
$$

Çünkü her set elemanı diğer her elemana bakar.

ISAB, bunu azaltmak için `m` tane öğrenilebilir **inducing point** kullanır:

$$
I = \{i_1, \ldots, i_m\}
$$

Genellikle:

$$
m \ll n
$$

ISAB iki aşamalıdır:

$$
H = \mathrm{MAB}(I, X)
$$

$$
\mathrm{ISAB}(X) = \mathrm{MAB}(X, H)
$$

Sezgisel akış:

```text
Input set X
     ↓
Inducing points I, input set X'e attention yapar
H = MAB(I, X)
     ↓
Input set X, bu induced summary H'ye attention yapar
ISAB(X) = MAB(X, H)
```

İlk adımda inducing point'ler set'ten özet bilgi çeker. İkinci adımda set elemanları bu özetlerden bilgi alır.


In [6]:
class ISAB(nn.Module):
    """
    Induced Set Attention Block.

    H = MAB(I, X)
    O = MAB(X, H)

    I: learnable inducing points, shape [m, d_model]
    """
    def __init__(self, d_model: int, num_heads: int, num_inducing_points: int, ff_hidden_dim: int = None):
        super().__init__()
        self.num_inducing_points = num_inducing_points
        self.inducing_points = nn.Parameter(torch.randn(num_inducing_points, d_model))
        self.mab1 = MAB(d_model, num_heads, ff_hidden_dim)
        self.mab2 = MAB(d_model, num_heads, ff_hidden_dim)

    def forward(self, X: torch.Tensor, return_attention: bool = False):
        batch_size = X.size(0)
        I = self.inducing_points.unsqueeze(0).expand(batch_size, -1, -1)

        if return_attention:
            H, w1 = self.mab1(I, X, return_attention=True)
            O, w2 = self.mab2(X, H, return_attention=True)
            return O, (w1, w2)

        H = self.mab1(I, X)
        O = self.mab2(X, H)
        return O

In [7]:
# ISAB shape testi
m = 3
isab = ISAB(d_model=d_model, num_heads=num_heads, num_inducing_points=m)

out, (w1, w2) = isab(X, return_attention=True)
print("X shape:       ", X.shape)
print("ISAB out shape:", out.shape)
print("Step 1 weights:", w1.shape, "# I attends to X")
print("Step 2 weights:", w2.shape, "# X attends to H")

X shape:        torch.Size([2, 5, 16])
ISAB out shape: torch.Size([2, 5, 16])
Step 1 weights: torch.Size([2, 4, 3, 5]) # I attends to X
Step 2 weights: torch.Size([2, 4, 5, 3]) # X attends to H


## 6. PMA: Pooling by Multihead Attention

Paper'daki bölümün ana fikri şudur:

> Mean/max pooling yerine, öğrenilebilir seed vector'ler kullanarak attention tabanlı pooling yapalım.

Klasik pooling yöntemleri:

```text
mean pooling: bütün elemanlara eşit ağırlık verir
max pooling: feature bazında maksimumu alır
```

Ama birçok problemde her elemanın hedef üzerindeki etkisi eşit değildir.

PMA formülü:

$$
\mathrm{PMA}_k(Z) = \mathrm{MAB}(S, \mathrm{rFF}(Z))
$$

Burada:

- $$Z \in \mathbb{R}^{n \times d}$$ encoder'dan gelen set feature'larıdır.
- $$S \in \mathbb{R}^{k \times d}$$ öğrenilebilir seed vector set'idir.
- `k`, kaç tane output representation istediğimizi belirler.

MAB açısından:

$$
Q = S,
\quad K = \mathrm{rFF}(Z),
\quad V = \mathrm{rFF}(Z)
$$

Yani seed vector'ler query olur, input set feature'ları key/value olur.

Sezgisel olarak:

> Seed vector'ler set'e soru sorar; set elemanları cevap verir; attention hangi elemanların önemli olduğunu belirler.


In [8]:
class PMA(nn.Module):
    """
    Pooling by Multihead Attention.

    PMA_k(Z) = MAB(S, rFF(Z))

    S: learnable seed vectors, shape [k, d_model]
    Z: encoder output, shape [batch, n, d_model]

    Output shape:
        [batch, k, d_model]
    """
    def __init__(self, d_model: int, num_heads: int, num_seeds: int, ff_hidden_dim: int = None):
        super().__init__()
        self.num_seeds = num_seeds
        self.seed_vectors = nn.Parameter(torch.randn(num_seeds, d_model))
        self.rff = RowWiseFeedForward(d_model, ff_hidden_dim)
        self.mab = MAB(d_model, num_heads, ff_hidden_dim)

    def forward(self, Z: torch.Tensor, return_attention: bool = False):
        batch_size = Z.size(0)
        S = self.seed_vectors.unsqueeze(0).expand(batch_size, -1, -1)
        U = self.rff(Z)
        return self.mab(S, U, return_attention=return_attention)

In [9]:
# PMA shape testi
k = 1
pma = PMA(d_model=d_model, num_heads=num_heads, num_seeds=k)

pooled, pma_weights = pma(out, return_attention=True)
print("Encoder output Z shape:", out.shape)
print("PMA output shape:      ", pooled.shape)
print("PMA attention weights: ", pma_weights.shape)  # [batch, heads, k, n]

Encoder output Z shape: torch.Size([2, 5, 16])
PMA output shape:       torch.Size([2, 1, 16])
PMA attention weights:  torch.Size([2, 4, 1, 5])


## 7. PMA ile mean pooling farkı

Mean pooling:

$$
z_{pool} = \frac{1}{n}\sum_{i=1}^n z_i
$$

Bu bütün elemanlara eşit ağırlık verir.

PMA ise attention ağırlıkları üretir:

$$
\alpha_i = \mathrm{softmax}(q k_i^T)
$$

$$
z_{pool} = \sum_{i=1}^n \alpha_i v_i
$$

Yani PMA'nın ağırlıkları input'a ve öğrenilmiş seed query'ye bağlıdır.

Mean pooling özel durumda şuna benzer:

$$
\alpha_1 = \alpha_2 = \cdots = \alpha_n = \frac{1}{n}
$$

PMA ise şunu öğrenebilir:

```text
önemli eleman      → yüksek attention
önemsiz elemanlar  → düşük attention
```


In [10]:
# Mean pooling ve PMA output shape karşılaştırması
Z = torch.randn(batch_size, n, d_model)

mean_pool = Z.mean(dim=1, keepdim=True)  # [batch, 1, d_model]
pma_pool, weights = pma(Z, return_attention=True)

print("Z shape:         ", Z.shape)
print("Mean pool shape: ", mean_pool.shape)
print("PMA pool shape:  ", pma_pool.shape)
print("Attention weights over set elements:", weights.shape)

# Ortalama attention dağılımını görelim: batch=0, head'lerin ortalaması, seed=0
avg_weights = weights[0, :, 0, :].mean(dim=0)
print("Average PMA attention weights for sample 0:")
print(avg_weights)
print("Sum:", avg_weights.sum().item())

Z shape:          torch.Size([2, 5, 16])
Mean pool shape:  torch.Size([2, 1, 16])
PMA pool shape:   torch.Size([2, 1, 16])
Attention weights over set elements: torch.Size([2, 4, 1, 5])
Average PMA attention weights for sample 0:
tensor([0.2000, 0.2027, 0.1963, 0.2022, 0.1988], grad_fn=<MeanBackward1>)
Sum: 0.9999999403953552


## 8. Basit sezgi: maksimum değeri bulma

Paper şu sezgiyi verir:

Eğer hedef, sayı setindeki maksimum değerse, aslında bütün elemanlar eşit önemli değildir. Sadece en büyük eleman kritik olabilir.

Örnek:

$$
X = \{1.2, 0.4, 7.8, 2.1\}
$$

Target:

$$
\max(X) = 7.8
$$

Mean pooling maksimumu doğrudan yakalamaz:

$$
\mathrm{mean}(X) = 2.875
$$

PMA ise öğrenebilirse en büyük elemana yüksek attention verebilir:

```text
1.2 → düşük attention
0.4 → düşük attention
7.8 → yüksek attention
2.1 → düşük attention
```


In [11]:
# Bu kısım eğitilmiş bir model değil; sadece attention pooling'in nasıl çalıştığını göstermek için
# elle tanımlanmış bir query ile en büyük değere daha fazla ağırlık vermeyi simüle ediyoruz.

values = torch.tensor([[1.2, 0.4, 7.8, 2.1]])  # [batch=1, n=4]

# Skorları doğrudan değerlere göre üretirsek softmax en büyük elemana ağırlık verir.
scores = values
weights = F.softmax(scores, dim=-1)
weighted_sum = (weights * values).sum(dim=-1)

print("Values:      ", values)
print("Softmax attn:", weights)
print("Weighted sum:", weighted_sum)
print("Max value:   ", values.max(dim=-1).values)

Values:       tensor([[1.2000, 0.4000, 7.8000, 2.1000]])
Softmax attn: tensor([[1.3532e-03, 6.0802e-04, 9.9471e-01, 3.3283e-03]])
Weighted sum: tensor([7.7676])
Max value:    tensor([7.8000])


## 9. `k` seed vector ne işe yarar?

PMA'nın output sayısı seed sayısı kadardır:

$$
\mathrm{PMA}_k(Z) \rightarrow \{o_1, o_2, \ldots, o_k\}
$$

- `k = 1`: bütün set'ten tek representation çıkarılır.
- `k > 1`: set'ten birden fazla output/prototype/cluster-like representation çıkarılır.

Örneğin clustering için:

```text
Input points X
     ↓
PMA with k=3 seed vectors
     ↓
{o1, o2, o3}
```

Bu üç output, üç cluster/prototype temsilini öğrenebilir.


In [12]:
# k seed ile PMA
k = 3
pma3 = PMA(d_model=d_model, num_heads=num_heads, num_seeds=k)

Z = torch.randn(batch_size, n, d_model)
O, weights = pma3(Z, return_attention=True)

print("Z shape:          ", Z.shape)
print("PMA_k output shape:", O.shape, "# [batch, k, d_model]")
print("Weights shape:    ", weights.shape, "# [batch, heads, k, n]")

# sample 0, head ortalaması için her seed'in input elemanlarına attention dağılımı
avg_weights = weights[0].mean(dim=0)  # [k, n]
print("Average attention per seed for sample 0:")
print(avg_weights)
print("Row sums:", avg_weights.sum(dim=-1))

Z shape:           torch.Size([2, 5, 16])
PMA_k output shape: torch.Size([2, 3, 16]) # [batch, k, d_model]
Weights shape:     torch.Size([2, 4, 3, 5]) # [batch, heads, k, n]
Average attention per seed for sample 0:
tensor([[0.1998, 0.1983, 0.2024, 0.1994, 0.2001],
        [0.2041, 0.1945, 0.2001, 0.2013, 0.1999],
        [0.2060, 0.1960, 0.1993, 0.2055, 0.1931]], grad_fn=<MeanBackward1>)
Row sums: tensor([1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


## 10. PMA sonrası SAB neden kullanılır?

Paper şunu öneriyor:

$$
H = \mathrm{SAB}(\mathrm{PMA}_k(Z))
$$

Yani önce `k` tane pooled output üretilir, sonra bu `k` output kendi aralarında self-attention yapar.

Neden?

Çünkü `k` output birbirinden bağımsız olmamalı.

Clustering örneğinde:

```text
cluster output 1
cluster output 2
cluster output 3
```

birbirleriyle koordineli olmalıdır. İki output aynı cluster'a odaklanırsa kötü olur.

PMA sonrası SAB, outputlar arasında etkileşim kurar:

```text
o1 ↔ o2 ↔ o3
```

Bu, paper'daki **explaining-away** fikrini destekler:

> Bir output bir pattern'i açıkladıysa, diğer outputların aynı pattern'i tekrar açıklamasına gerek kalmaz.


In [13]:
class SetDecoder(nn.Module):
    """
    Set Transformer decoder örneği:
        PMA_k(Z) -> SAB -> rFF/output head için representation
    """
    def __init__(self, d_model: int, num_heads: int, num_outputs: int):
        super().__init__()
        self.pma = PMA(d_model, num_heads, num_outputs)
        self.sab = SAB(d_model, num_heads)

    def forward(self, Z: torch.Tensor, return_attention: bool = False):
        if return_attention:
            O, pma_w = self.pma(Z, return_attention=True)
            H, sab_w = self.sab(O, return_attention=True)
            return H, (pma_w, sab_w)

        O = self.pma(Z)
        H = self.sab(O)
        return H


decoder = SetDecoder(d_model=d_model, num_heads=num_heads, num_outputs=3)
H, (pma_w, sab_w) = decoder(Z, return_attention=True)

print("Z shape:          ", Z.shape)
print("Decoder out shape:", H.shape)
print("PMA weights:      ", pma_w.shape, "# seeds attend to input set")
print("SAB weights:      ", sab_w.shape, "# outputs attend to each other")

Z shape:           torch.Size([2, 5, 16])
Decoder out shape: torch.Size([2, 3, 16])
PMA weights:       torch.Size([2, 4, 3, 5]) # seeds attend to input set
SAB weights:       torch.Size([2, 4, 3, 3]) # outputs attend to each other


## 11. Komple Set Transformer örneği

Tipik bir set-level classification/regression modeli şu şekilde kurulabilir:

```text
Input set X
     ↓
Set Encoder: SAB/ISAB layers
     ↓
Z
     ↓
PMA(k=1)
     ↓
Linear head
     ↓
Prediction
```

Eğer element-level prediction istiyorsan PMA kullanmadan encoder output'unu doğrudan her elemana uygulayabilirsin.

Eğer set-level output istiyorsan PMA kullanmak mantıklıdır.


In [14]:
class SetTransformerClassifier(nn.Module):
    """
    Basit set-level classifier.

    Input shape:
        X: [batch, n, input_dim]

    Output shape:
        logits: [batch, num_classes]
    """
    def __init__(
        self,
        input_dim: int,
        d_model: int,
        num_heads: int,
        num_classes: int,
        num_inducing_points: int = 8,
        use_isab: bool = True,
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)

        if use_isab:
            self.encoder = nn.Sequential(
                ISAB(d_model, num_heads, num_inducing_points),
                ISAB(d_model, num_heads, num_inducing_points),
            )
        else:
            self.encoder = nn.Sequential(
                SAB(d_model, num_heads),
                SAB(d_model, num_heads),
            )

        self.pool = PMA(d_model, num_heads, num_seeds=1)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, num_classes),
        )

    def forward(self, X: torch.Tensor):
        H = self.input_proj(X)
        H = self.encoder(H)
        pooled = self.pool(H)       # [batch, 1, d_model]
        pooled = pooled.squeeze(1)  # [batch, d_model]
        logits = self.head(pooled)
        return logits

In [15]:
# Komple model shape testi
batch_size = 4
n = 10
input_dim = 6
num_classes = 3

X = torch.randn(batch_size, n, input_dim)
model = SetTransformerClassifier(
    input_dim=input_dim,
    d_model=32,
    num_heads=4,
    num_classes=num_classes,
    num_inducing_points=5,
    use_isab=True,
)

logits = model(X)
print("Input X shape:", X.shape)
print("Logits shape: ", logits.shape)

Input X shape: torch.Size([4, 10, 6])
Logits shape:  torch.Size([4, 3])


## 12. Permutation invariance testi

Set-level modelde input set'in sırasını değiştirdiğinde output'un aynı kalmasını isteriz.

Teoride SAB/ISAB + PMA bu özelliği destekler. Ancak floating point işlemler, dropout gibi şeyler küçük farklar yaratabilir. Burada dropout yok, bu yüzden outputlar çok yakın olmalı.


In [16]:
model.eval()

X = torch.randn(1, 8, input_dim)
perm = torch.randperm(X.size(1))
X_perm = X[:, perm, :]

with torch.no_grad():
    y1 = model(X)
    y2 = model(X_perm)

print("Permutation:", perm.tolist())
print("Output original: ", y1)
print("Output permuted: ", y2)
print("Max abs diff:    ", (y1 - y2).abs().max().item())

Permutation: [1, 6, 2, 7, 4, 5, 3, 0]
Output original:  tensor([[0.1633, 0.0990, 0.0233]])
Output permuted:  tensor([[0.1633, 0.0990, 0.0233]])
Max abs diff:     1.4901161193847656e-08


## 13. Özet

Set Transformer'daki temel fikirleri şöyle toparlayabiliriz:

| Blok | Tanım | Mantık |
|---|---|---|
| `MAB(X, Y)` | Multihead Attention Block | `X`, `Y`'den bilgi çeker |
| `SAB(X)` | `MAB(X, X)` | Set içi self-attention |
| `ISAB(X)` | `MAB(I, X)` sonra `MAB(X, H)` | Inducing point ile verimli attention |
| `PMA_k(Z)` | `MAB(S, rFF(Z))` | Seed vector'ler ile attention pooling |

PMA'nın ana katkısı:

```text
Mean/max pooling yerine öğrenilebilir attention pooling.
```

PMA'da:

$$
Q = S,
\quad K = \mathrm{rFF}(Z),
\quad V = \mathrm{rFF}(Z)
$$

Sonuç:

$$
n \text{ input element} \rightarrow k \text{ output representation}
$$

- `k = 1`: set-level classification/regression için tek temsil
- `k > 1`: clustering/prototype/multiple-output görevleri için çoklu temsil

PMA sonrası SAB:

```text
k output'un birbirleriyle etkileşmesini sağlar.
```

Bu özellikle clustering gibi outputlar arasında koordinasyon gerektiren görevlerde faydalıdır.
